# Testing Functionality of CPU and GPU with Tensorflow

In [ ]:
import tensorflow as tf
import time

# Check if GPU is detecting on device or not
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# Test CPU vs GPU performance
cpu = tf.device('/CPU:0')
gpu = tf.device('/GPU:0')

# Large matrix multiplication
matrix_size = (10000, 10000)

def benchmark(device):
    with device:
        a = tf.random.normal(matrix_size)
        b = tf.random.normal(matrix_size)
        start = time.time()
        c = tf.matmul(a, b)  # Matrix multiplication
        tf.experimental.numpy.sum(c)  # Force computation
        end = time.time()
        print(f"Computation time on {device}: {end - start:.4f} seconds")

print("Benchmarking TensorFlow computation...")
benchmark(cpu)
benchmark(gpu)


# Installing the Required Dependency (Run if required, here as in Conda Environment, all are already setup in Root folder)


In [ ]:
!pip install tensorflow keras pandas numpy scikit-learn matplotlib seaborn transformers

# Import required Libraries and Modules

In [ ]:

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Imports done") # To check if everything is imported successfully and Cell working is done.

# Importing data from the CSV file in root folder

In [ ]:
data = pd.read_csv("data.csv")  # Load Data

# Display first 5 rows to check for corrert format of data 
print("First 5 rows of raw data:\n", data.head())

# Check for the Data Sanitation

In [ ]:
# Inspect the original label column'Attrition' values to see what we're working with.
print("Original Attrition values:", data['Attrition'].unique())

In [ ]:
# Clean the 'Attrition' column: strip whitespace and convert to lowercase
data['Attrition'] = data['Attrition'].astype(str).str.strip().str.lower()

# Inspect cleaned values
print("Cleaned Attrition values:", data['Attrition'].unique())

In [ ]:
mapping = {'no': 0, 'yes': 1}

# Map the cleaned values to numeric values
data['Attrition'] = data['Attrition'].map(mapping)

# Check the mapping result
print("Mapped Attrition values:", data['Attrition'].unique())
                                          

In [ ]:
# Check for missing values
print("Missing Values in Each Column:\n", data.isnull().sum()) 

In [ ]:

# Check target class balance
print("\nClass Distribution (Attrition):\n", data["Attrition"].value_counts()) 

In [ ]:
# Drop non-informative/identifier columns
drop_cols = ['Over18', 'EmployeeCount', 'EmployeeNumber', 'StandardHours']
data.drop(columns=drop_cols, inplace=True)

# Define your numerical and categorical columns.
numerical_cols = [
    "Age", "DailyRate", "DistanceFromHome", "Education", "EnvironmentSatisfaction", "HourlyRate", 
    "JobInvolvement", "JobLevel", "JobSatisfaction", "MonthlyIncome", "MonthlyRate", "NumCompaniesWorked", 
    "PercentSalaryHike", "PerformanceRating", "RelationshipSatisfaction", "StockOptionLevel", 
    "TotalWorkingYears", "TrainingTimesLastYear", "WorkLifeBalance", "YearsAtCompany", "YearsInCurrentRole",
    "YearsSinceLastPromotion", "YearsWithCurrManager"
]

categorical_cols = [
    "BusinessTravel", "Department", "EducationField", "Gender", "JobRole",
    "MaritalStatus", "OverTime"
]

target_col = "Attrition"

print("Columns are defined expliclitly")

# check for Error casuing values 

for col in categorical_cols:
    print(f"Unique values in {col}:", data[col].unique())

for col in numerical_cols:
    print(f"Unique values in {col}:", data[col].unique())


    


# After Analysis this was previous output 

```
Python

Unique values in BusinessTravel: ['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
Unique values in Department: ['Sales' 'Research & Development' 'Human Resources']
Unique values in EducationField: ['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
Unique values in Gender: ['Female' 'Male']
Unique values in JobRole: ['Sales Executive' 'Research Scientist' 'Laboratory Technician'
 'Manufacturing Director' 'Healthcare Representative' 'Manager'
 'Sales Representative' 'Research Director' 'Human Resources']
Unique values in MaritalStatus: ['Single' 'Married' 'Divorced']
Unique values in OverTime: ['Yes' 'No']
Unique values in Over18: ['Y']
Unique values in Age: [41 49 37 33 27 32 59 30 38 36 35 29 31 34 28 22 53 24 21 42 44 46 39 43
 50 26 48 55 45 56 23 51 40 54 58 20 25 19 57 52 47 18 60]
Unique values in DailyRate: [1102  279 1373 ... 1156  398 1061]
Unique values in DistanceFromHome: [ 1  8  2  3 24 23 27 16 15 26 19 21  5 11  9  7  6 10  4 25 12 18 29 22
 14 20 28 17 13]
Unique values in Education: [2 1 4 3 5]
Unique values in EmployeeCount: [1]
Unique values in EmployeeNumber: [   1    2    4 ... 2566 2567 2568]
Unique values in EnvironmentSatisfaction: [2 3 4 1]
Unique values in HourlyRate: [ 94  61  92  56  40  79  81  67  44  84  49  31  93  50  51  80  96  78
  45  82  53  83  58  72  48  42  41  86  97  75  33  37  73  98  36  47
  71  30  43  99  59  95  57  76  87  66  55  32  52  70  62  64  63  60
 100  46  39  77  35  91  54  34  90  65  88  85  89  68  69  74  38]
Unique values in JobInvolvement: [3 2 4 1]
Unique values in JobLevel: [2 1 3 4 5]
Unique values in JobSatisfaction: [4 2 3 1]
Unique values in MonthlyIncome: [5993 5130 2090 ... 3409 9851 2512]
Unique values in MonthlyRate: [19479 24907  2396 ... 13398  9676 25694]
Unique values in NumCompaniesWorked: [8 1 6 9 0 4 5 2 7 3]
Unique values in PercentSalaryHike: [11 23 15 12 13 20 22 21 17 14 16 18 19 24 25]
Unique values in PerformanceRating: [3 4]
Unique values in RelationshipSatisfaction: [1 4 2 3]
Unique values in StandardHours: [80]
Unique values in StockOptionLevel: [0 1 3 2]
Unique values in TotalWorkingYears: [ 8 10  7  6 12  1 17  5  3 31 13  0 26 24 22  9 19  2 23 14 15  4 29 28
 21 25 20 11 16 37 38 30 40 18 36 34 32 33 35 27]
Unique values in TrainingTimesLastYear: [0 3 2 5 1 4 6]
Unique values in WorkLifeBalance: [1 3 2 4]
Unique values in YearsAtCompany: [ 6 10  0  8  2  7  1  9  5  4 25  3 12 14 22 15 27 21 17 11 13 37 16 20
 40 24 33 19 36 18 29 31 32 34 26 30 23]
Unique values in YearsInCurrentRole: [ 4  7  0  2  5  9  8  3  6 13  1 15 14 16 11 10 12 18 17]
Unique values in YearsSinceLastPromotion: [ 0  1  3  2  7  4  8  6  5 15  9 13 12 10 11 14]
Unique values in YearsWithCurrManager: [ 5  7  0  2  6  8  3 11 17  1  4 12  9 10 15 13 16 14]
```

based on it following Non-Informative or Identifier Columns are dropped -
Over18: Observation: Only one unique value ("Y").

EmployeeCount:Observation: Constant (always 1).

EmployeeNumber:Observation: Likely an identifier with many unique values but no predictive power.

StandardHours:Observation: Constant (80).

Dropped this columns since they doesn’t provide any variability.


In [ ]:
# Split features and target
X = data.drop(target_col, axis=1)
y = data[target_col]

# Set up a ColumnTransformer that applies scaling to numerical columns and one–hot encodes categorical columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), categorical_cols)
      #  drop="first"


    ],
    # remainder="passthrough"  # Optionally, pass through any columns not explicitly mentioned.
)

# Fit and transform the features
X_processed = preprocessor.fit_transform(X)
print("Processed feature shape:", X_processed.shape)



# Optionally, review the unique values for categorical columns if needed
for col in categorical_cols:
    print(f"{col}: {data[col].nunique()} unique values")

# Train–validation split
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)





# Here We Can Apply some methods in like - SMOTE, or weighting or focal loss

#  --- Part 1: Variational Autoencoder (VAE) with Attention-Based Feature Extraction ---

In [ ]:
# Sampling layer for reparameterization trick
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

def create_vae(input_dim, latent_dim=74):
    # Encoder
    input_layer = keras.Input(shape=(input_dim,), name='encoder_input')
    x = layers.Dense(128, activation='relu')(input_layer)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(96, activation='relu')(x)
    # Instead of a direct latent vector, we calculate distribution parameters
    z_mean = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
    z = Sampling()([z_mean, z_log_var])
    encoder = keras.Model(input_layer, [z_mean, z_log_var, z], name='encoder')

    # Decoder
    latent_inputs = keras.Input(shape=(latent_dim,), name='z_sampling')
    x = layers.Dense(96, activation='relu')(latent_inputs)
    x = layers.Dense(128, activation='relu')(x)
    decoder_outputs = layers.Dense(input_dim, activation='linear')(x)
    decoder = keras.Model(latent_inputs, decoder_outputs, name='decoder')

    # VAE Model: map input to reconstruction through latent space
    outputs = decoder(z)
    vae = keras.Model(input_layer, outputs, name='vae')

    # Compute reconstruction loss and KL divergence loss
    reconstruction_loss = keras.losses.mse(input_layer, outputs)
    reconstruction_loss *= input_dim
    kl_loss = 1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
    kl_loss = tf.reduce_sum(kl_loss, axis=-1) * -0.5
    vae_loss = tf.reduce_mean(reconstruction_loss + kl_loss)
    vae.add_loss(vae_loss)
    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
    
    return vae, encoder, decoder
print("Done Variational Autoencoder (VAE) with Attention-Based Feature Extraction")

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
import os
import psutil
import lightgbm as lgb

# Set the number of physical CPU cores
os.environ["LOKY_MAX_CPU_COUNT"] = str(psutil.cpu_count(logical=False))

# Define LightGBM parameters with L2 regularization
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "lambda_l2": 10,  # Increase to apply stronger regularization
    "n_jobs": psutil.cpu_count(logical=False),  # Use physical cores
}





In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from tensorflow.keras.activations import swish

# Load and split the dataset
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# VAE Model
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# VAE Encoder
latent_dim = 74
input_dim = X_train.shape[1]
inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(128, activation=swish)(inputs)
x = layers.Dense(96, activation=swish)(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(inputs, [z_mean, z_log_var, z], name='encoder')

# VAE Decoder
decoder_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(96, activation=swish)(decoder_inputs)
x = layers.Dense(128, activation=swish)(x)
outputs = layers.Dense(input_dim, activation='linear')(x)
decoder = keras.Model(decoder_inputs, outputs, name='decoder')

# VAE Loss
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstructed = self.decoder(z)
        kl_loss = -0.5 * tf.reduce_mean(z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)
        self.add_loss(kl_loss)
        return reconstructed

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4), loss='mse')
vae.fit(X_train, X_train, validation_data=(X_val, X_val), epochs=100, batch_size=128, callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)], verbose=1)

# Extract latent features
X_train_latent = encoder.predict(X_train)[2]
X_val_latent = encoder.predict(X_val)[2]

# Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation=swish),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
    
    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# LGBM Classifier
# lgbm = LGBMClassifier()
lgbm = lgb.LGBMClassifier(**params)
lgbm.fit(X_train_latent, y_train)
lgbm_preds = lgbm.predict_proba(X_val_latent)[:, 1]

# MLP + Transformer Ensemble Model
latent_input = keras.Input(shape=(latent_dim,), name="latent_input")
mlp_branch = layers.Dense(64, activation=swish)(latent_input)
mlp_branch = layers.Dropout(0.5)(mlp_branch)
mlp_branch = layers.Dense(32, activation=swish)(mlp_branch)

transformer_branch = layers.Reshape((latent_dim, 1), name="reshape_for_transformer")(latent_input)
transformer_branch = TransformerBlock(embed_dim=32, num_heads=8, ff_dim=128, rate=0.1)(transformer_branch)
transformer_branch = layers.Flatten(name="flatten_transformer")(transformer_branch)

merged = layers.Concatenate(name="concatenate_branches")([mlp_branch, transformer_branch])
x = layers.Dense(64, activation=swish)(merged)
x = layers.Dropout(0.5)(x)
x = layers.Dense(32, activation=swish)(x)
output = layers.Dense(1, activation='sigmoid', name="output")(x)

ensemble_model = keras.Model(inputs=latent_input, outputs=output, name="Ensemble_Model")
ensemble_model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4), loss="binary_crossentropy", metrics=["accuracy"])
ensemble_model.fit(X_train_latent, y_train, validation_data=(X_val_latent, y_val), epochs=50, batch_size=128, callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)], verbose=1)

# Evaluate
val_loss, val_accuracy = ensemble_model.evaluate(X_val_latent, y_val, verbose=0)
print(f"Validation Accuracy: {val_accuracy:.4f}")


# 2nd Approach

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Reshape, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ---- Step 1: Adjust VAE ----
latent_dim = 64  # Increased latent space
encoder = tf.keras.Sequential([
    Dense(256, activation="relu"),  # Increased size of first layer
    Dense(128, activation="relu"),  # Increased size of second layer
    Dense(latent_dim, activation="relu")  # Using ReLU instead of linear
])

# ---- Step 2: Improve Transformer & MLP ----
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads=12, ff_dim=128, rate=0.2):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization()
        self.layernorm2 = tf.keras.layers.LayerNormalization()
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# ---- Step 3: Define Model ----
input_layer = Input(shape=(X_train.shape[1],))  # Adjust input shape to match your data
encoded = encoder(input_layer)

# Reshape for Transformer (if required)
transformer_input = Reshape((latent_dim, 1))(encoded)  # Reshape to (latent_dim, 1)
transformer_output = TransformerBlock(embed_dim=latent_dim)(transformer_input)

# Flatten Transformer Output and Combine with MLP
flattened = Flatten()(transformer_output)
final_representation = tf.keras.layers.Concatenate()([encoded, flattened])  # Merge latent + transformer

# ---- MLP Branch ----
mlp = tf.keras.Sequential([
    Dense(256, activation="relu"),  # Increased size
    BatchNormalization(),
    Dropout(0.3),  # Increased dropout
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

output_layer = mlp(final_representation)

# Create Model
model = Model(inputs=input_layer, outputs=output_layer)

# ---- Step 4: Adjust Learning Rate & Optimizer ----
optimizer = Adam(learning_rate=1e-5)  # Lower LR for better convergence

# ---- Step 5: LGBM with Stronger Regularization ----
lgbm_params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "metric": "binary_logloss",
    "bagging_fraction": 0.8,
    "lambda_l2": 1.0,  # Increased L2 regularization
    "learning_rate": 0.05,  # Lower learning rate
    "num_leaves": 50,  # Increased number of leaves
      # Limit tree depth to control overfitting
    "feature_fraction": 0.8  # Use a fraction of features per tree to reduce overfitting
    
}

# lgbm_params = {
#     "objective": "binary",
#     "boosting_type": "gbdt",
#     "metric": "binary_logloss",
#     "bagging_fraction": 0.8,
#     "feature_fraction": 0.8,
#     "lambda_l2": 1.0,
#     "learning_rate": 0.05,
#     "num_leaves": 50,
#     "max_depth": 10,
# }

lgbm = LGBMClassifier(**lgbm_params)

# ---- Step 6: Compute Class Weights ----
classes = np.unique(y_train)
class_weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = {i: class_weights[i] for i in range(len(classes))}

# ---- Model Compilation ----
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

# ---- Early Stopping ----
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# ---- Train the Model ----
model.fit(X_train, y_train, epochs=50, batch_size=32, class_weight=class_weight_dict,
          validation_data=(X_val, y_val), callbacks=[early_stopping])

# ---- Train LGBM ----
lgbm.fit(X_train, y_train, sample_weight=[class_weight_dict[label] for label in y_train])

# ---- Evaluate ----
nn_acc = model.evaluate(X_val, y_val)
lgbm_acc = lgbm.score(X_val, y_val)

print(f"Neural Network Accuracy: {nn_acc[1]:.4f}")
print(f"LGBM Accuracy: {lgbm_acc:.4f}")


In [ ]:
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

# --- Neural Network Model ---
# Define the neural network model architecture
nn_model = models.Sequential([
    layers.InputLayer(input_shape=(51,)),  # Input layer (flattened)
    layers.Dense(128, activation='relu', kernel_regularizer='l2'),  # Hidden layer with L2 regularization
    layers.Dropout(0.3),  # Dropout to prevent overfitting
    layers.Dense(64, activation='relu', kernel_regularizer='l2'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu', kernel_regularizer='l2'),
    layers.Dense(1, activation='sigmoid')  # Output layer (binary classification)
])

# Learning rate scheduler function
def lr_schedule(epoch):
    if epoch < 10:
        return 0.001  # Initial learning rate
    else:
        return 0.0005  # Reduce learning rate after 10 epochs

# Compile the neural network model
optimizer = optimizers.Adam(learning_rate=0.001)
nn_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping callback for the neural network
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the neural network model with learning rate schedule and early stopping
history = nn_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, 
                       batch_size=32, callbacks=[LearningRateScheduler(lr_schedule), early_stopping])


# --- LightGBM Model ---
# Define LightGBM hyperparameters
params = {
    'objective': 'binary',         # Binary classification task
    'metric': 'binary_error',       # Error metric
    'num_leaves': 31,               # Control complexity to avoid overfitting
    'learning_rate': 0.01,          # Lower learning rate for stability
    'feature_fraction': 0.8,        # Use 80% of features at each iteration
    'bagging_fraction': 0.8,        # Use 80% of data at each iteration
    'bagging_freq': 5,              # Perform bagging every 5 iterations
    'lambda_l2': 1.0,               # L2 regularization
    'max_depth': -1,                # No depth limit
    'min_data_in_leaf': 20,         # Minimum data in leaf to avoid overfitting
    'boosting_type': 'gbdt',        # Gradient Boosting Decision Tree
    'verbose': -1                 # Suppress verbose output
}

# Prepare LightGBM training dataset
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# Train the LightGBM model with early stopping using a callback
lgb_model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_val],
    valid_names=['validation'],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(stopping_rounds=100)]
)

# Predict and evaluate the LightGBM model
y_pred_lgb = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
y_pred_binary_lgb = (y_pred_lgb > 0.5).astype(int)

# Evaluate LightGBM accuracy
accuracy_lgb = accuracy_score(y_val, y_pred_binary_lgb)
print(f"LGBM Model Accuracy: {accuracy_lgb:.4f}")


# --- Hyperparameter Tuning for LightGBM ---
# Grid search for hyperparameters in LightGBM
param_grid = {
    'num_leaves': [31, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'feature_fraction': [0.8, 0.9, 1.0],
    'bagging_fraction': [0.8, 0.9, 1.0],
    'lambda_l2': [0, 0.5, 1.0]
}

# Grid search on hyperparameters
grid_search = GridSearchCV(
    estimator=lgb.LGBMClassifier(objective='binary', metric='binary_error'),
    param_grid=param_grid,
    cv=5,
    verbose=1
)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
